# Wikimedia Silver Layer Pipeline

### Purpose
This notebook implements the Silver layer of a Wikimedia data pipeline using a medallion (Bronze → Silver) architecture.\
It reads raw Wikimedia edit events from a Bronze Delta table, deduplicates them, and upserts them into a Silver Delta table.

### Core streaming upsert logic 
- defines merge_to_silver() which deduplicates micro-batches by (wiki, title) keeping the latest event, then MERGEs into Silver. 
- A readStream reads from Bronze, filters out bots and non-edit/new events, and writes via foreachBatch with trigger(availableNow=True)


In [0]:
%run ./params_config

### Create table

In [0]:
spark.sql(f"CREATE OR REPLACE TABLE {catalog}.{silver_schema}.{silver_table} ( id long, title string,  comment string, title_url string, timestamp  timestamp , user string,  wiki string)")


###  Enable cdf to produce feed table

```sql
ALTER TABLE {catalog}.{bronze_schema}.{bronze_table}
SET TBLPROPERTIES (delta.enableChangeDataFeed = true);
```

This enables Change Data Feed on the Bronze table so that row-level changes (inserts, updates, deletes) are captured and can be consumed downstream by the Silver layer pipeline.

In [0]:
def merge_to_silver(microBatchDF, batch_id):
    # dedupe within the microbatch: keep latest event per page key
    w = Window.partitionBy("wiki", "title").orderBy(col("timestamp").desc())

    deduped = (microBatchDF
               .withColumn("rn", row_number().over(w))
               .filter("rn = 1")
               .drop("rn"))

    deduped.createOrReplaceTempView("wikimedia")

    spark.sql(f"""
        MERGE INTO {catalog}.{silver_schema}.{silver_table} t
        USING wikimedia s
        ON t.wiki = s.wiki AND t.title = s.title

        WHEN MATCHED AND s.type = 'edit' AND t.timestamp < s.timestamp
          THEN UPDATE SET *

        WHEN NOT MATCHED
          THEN INSERT *
    """)

(spark.readStream
    .table(f"{catalog}.{bronze_schema}.{bronze_table}")
    .filter((col("bot") == False) & (col("type").isin("edit", "new")))
    .select("id", "title", "comment", "title_url", "timestamp", "type", "user", "wiki")
    .writeStream
    .trigger(availableNow=True)
    .foreachBatch(merge_to_silver)
    .option("checkpointLocation", f"{checkpoint}")
    .start()
    .awaitTermination())

In [0]:
%sql
describe history workspace.silver.wikimedia_silver

In [0]:
%sql
select count(*) from workspace.silver.wikimedia_silver